# Metric walkthrough with a real worked example

Companion to `TODO.md` item 1: readers can't build intuition for `conciseness`,
`distance_score`, `uniqueness_entropy`, `unk_rate`, `uncovered_rate`, `tree_complexity`,
`exact_rate`, `unique_rate`, `semantic_coverage` from formulas alone. This notebook picks
real mapped concepts, tokenizes them against real candidate lists (`T`), and reads each
metric straight off the printed context tree.

**What you can change** (see the "Parameters" cell below): which mapped concepts to
inspect (`CONCEPT_IDS`, or leave empty to auto-pick an easy/typical/hard example), the
candidate-list size (`K_EXAMPLE`), and which candidate lists to compare
(`CANDIDATE_LISTS`, any number of `(category, file_type)` pairs from
`greedy_tree_margin`/`baseline`).

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import pickle

import polars as pl
from IPython.display import Markdown, display

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.eval as eval
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
from src.graph_tokenizer_gd_tree_dev import drilldown

## Load graph, mapped concepts, and available candidate lists

In [ ]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = sorted(df_mapped["id"].unique().to_list())  # sorted (unlike notebooks 1-4) so the
# auto-picked worked examples below are reproducible across reruns, not order-dependent on
# polars' unique()
D = config.TokenizerParam().max_dist_candidate

all_candidates = drilldown.load_all_candidates()
adj = tokenizer.build_out_adjacency(combined_subgraphs)
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)

print("Available candidate lists -- (category, file_type) pairs to use in CANDIDATE_LISTS below:")
for category, file_types in all_candidates.items():
    print(f"  {category}: {sorted(file_types)}")

In [ ]:
def search_concepts(query: str, limit: int = 20):
    """Find mapped-concept ids by id or label substring, to fill in CONCEPT_IDS below."""
    q = query.lower()
    hits = [
        (cid, id_to_label.get(cid, cid))
        for cid in mapped_ids
        if q in str(cid).lower() or q in str(id_to_label.get(cid, "")).lower()
    ]
    return hits[:limit]

## Parameters

Edit these, then re-run every cell below.

In [ ]:
K_EXAMPLE = 2000  # candidate-list size |T| to inspect -- same k used for every list below,
# so the worked examples stay a fair comparison (see IMPLEMENTATION.md's notebook-4 convention).

CANDIDATE_LISTS = [
    ("greedy_tree_margin", "1.0"),
    ("baseline", "most_children"),
]  # any number of (category, file_type) pairs from the printed list above.

CONCEPT_IDS = []  # mapped-concept ids to walk through, e.g. ["319826005", "2781000175103"].
# Leave empty to auto-pick one easy / typical / hard example instead (see below).
# Use search_concepts("some text") to look up ids by label, e.g. search_concepts("urokinase").

## Build `T` / context trees / scores for each requested candidate list

In [ ]:
def aggregate_metrics(trees, T):
    # Same 8 eval.py metrics as eval.evaluate(), computed from context trees already built
    # below -- eval.evaluate() would rebuild them from scratch (~20-30s each on this graph),
    # wasted work since the per-concept worked examples need these same trees anyway.
    return {
        "conciseness": eval.conciseness(trees),
        "distance_score": eval.distance_score(trees, D),
        "uniqueness_entropy": eval.uniqueness_entropy(trees),
        "unk_rate": eval.unk_rate(trees),
        "uncovered_rate": eval.uncovered_rate(trees),
        "tree_complexity": eval.tree_complexity(trees),
        "exact_rate": eval.exact_rate(mapped_ids, T),
        "unique_rate": eval.unique_rate(trees),
    }


methods = {}
for category, file_type in CANDIDATE_LISTS:
    name = f"{category}/{file_type}"
    T = drilldown.get_candidate_ids(all_candidates, category, file_type, K_EXAMPLE)
    trees = eval.build_context_trees(mapped_ids, adj, T, D, id_to_label)
    S = tokenizer.compute_semantic_coverage(A, node_to_idx, T, D)
    scores = drilldown.get_all_concept_scores(trees, S, node_to_idx, mapped_ids, D)
    agg = aggregate_metrics(trees, T)
    agg["semantic_coverage"] = tokenizer.semantic_coverage_score(S, node_to_idx, mapped_ids)
    methods[name] = {"T": T, "trees": trees, "scores": scores, "agg": agg}

agg_table = pl.DataFrame(
    [{"method": name, "k": K_EXAMPLE, **m["agg"]} for name, m in methods.items()],
)
agg_table

## Resolve which concepts to walk through

If `CONCEPT_IDS` was left empty, auto-pick one easy / typical / hard example using the
*first* configured candidate list as the difficulty reference. Per `TODO.md` item 1:
`any(ctx.uncovered for ctx in _iter_contexts(tree))` (`eval.py:89`) is the cheap
per-concept `unk_rate` detail that buckets candidates by difficulty before picking
examples -- "easy" = no uncovered branch and the shallowest tree, "hard" = the highest
fraction of uncovered branches, "typical" = the median-sized fully-covered tree.

In [ ]:
def concept_stats(tree):
    contexts = list(eval._iter_contexts(tree))
    n_uncovered = sum(1 for ctx in contexts if ctx.uncovered)
    return {
        "n_contexts": len(contexts),
        "n_tokens": sum(len(ctx.tokens) for ctx in contexts),
        "has_unk": n_uncovered > 0,
        "uncovered_fraction": n_uncovered / len(contexts) if contexts else 0.0,
    }


if CONCEPT_IDS:
    missing = [c for c in CONCEPT_IDS if c not in mapped_ids]
    if missing:
        print(f"Warning: {missing} not found in mapped_ids -- dropping.")
    example_concepts = {f"Concept {i + 1}": c for i, c in enumerate(CONCEPT_IDS) if c in mapped_ids}
else:
    ref_trees = next(iter(methods.values()))["trees"]
    stats_ref = {c: concept_stats(ref_trees[c]) for c in mapped_ids}

    covered = sorted(
        (c for c in mapped_ids if not stats_ref[c]["has_unk"]),
        key=lambda c: (stats_ref[c]["n_contexts"], stats_ref[c]["n_tokens"]),
    )
    uncovered_pool = [c for c in mapped_ids if stats_ref[c]["has_unk"]]

    example_concepts = {
        "Easy case (shallow tree, fully covered)": covered[0] if covered else None,
        "Typical case (fully covered, more structure)": covered[len(covered) // 2] if covered else None,
        "Hard case (mostly uncovered branches)": (
            max(uncovered_pool, key=lambda c: (stats_ref[c]["uncovered_fraction"], stats_ref[c]["n_contexts"]))
            if uncovered_pool
            else None
        ),
    }
    example_concepts = {tag: c for tag, c in example_concepts.items() if c is not None}

for tag, cid in example_concepts.items():
    print(tag, "->", cid, id_to_label.get(cid, cid))

## Worked examples

For each concept above, and for every candidate list in `CANDIDATE_LISTS`: the tokenized
context tree (`•` = a token found, `∅` = a branch that died without one, IS_A hops stay
in the same branch, every other relation opens a nested one -- same rendering
`app_new_tokenizer.py`'s drill-down tab uses), then one line per metric tying its number
back to what's visible in the tree, next to that candidate list's aggregate over all
mapped concepts for comparison.

In [ ]:
def explain_concept(cid: str, tag: str):
    display(Markdown(f"### {tag}: {id_to_label.get(cid, cid)} (`{cid}`)"))

    for name, m in methods.items():
        tree = m["trees"][cid]
        row = m["scores"].filter(pl.col("mapped_id") == cid)
        agg = m["agg"]

        display(Markdown(f"**{name}** (k={K_EXAMPLE})"))
        print(drilldown.render_context_tree(tree, id_to_label))
        print(f"s-expr: {tree.to_sexpr()}")

        contexts = list(eval._iter_contexts(tree))
        distances = [d for ctx in contexts for _, d in ctx.tokens]
        n_uncovered = sum(1 for ctx in contexts if ctx.uncovered)
        n_tokens = sum(len(ctx.tokens) for ctx in contexts)
        n_contexts = len(contexts)
        is_exact = drilldown.is_exact_match(tree, cid)
        frac_sem_cov = row["frac_sem_cov"].item() if row.height else None
        dist_score = row["distance_score"].item() if row.height else None
        redundancy = row["redundancy_group_size"].item() if row.height else None

        dist_penalized = distances + [D + 1] * n_uncovered
        mean_dist_penalized = sum(dist_penalized) / len(dist_penalized) if dist_penalized else D + 1

        lines = [
            f"- **conciseness** — {n_tokens} token leaf/leaves in this tree "
            f"(method aggregate over all {len(mapped_ids):,} concepts: {agg['conciseness']:.2f}).",
            f"- **distance_score** — tokens found at hop distances {distances or '[]'}"
            + (f", plus {n_uncovered} uncovered branch(es) counted at D+1={D + 1}" if n_uncovered else "")
            + f" → mean {mean_dist_penalized:.2f} → contribution "
            + (f"{dist_score:.2f}" if dist_score is not None else "n/a")
            + f" via `1 - mean/(D+1)` (method aggregate: {agg['distance_score']:.2f}).",
            f"- **uniqueness_entropy** — this concept's context-tree signature is shared by "
            + (f"{int(redundancy)} concept(s) total" if redundancy is not None else "n/a")
            + (" (unique — maximally discriminative)" if redundancy == 1 else " (a collision — pulls the aggregate entropy down)")
            + f" (method aggregate entropy: {agg['uniqueness_entropy']:.2f}).",
            f"- **unique_rate** — this concept's signature "
            + (
                "is unique (redundancy_group_size = 1) → counts toward unique_rate"
                if redundancy == 1
                else f"is shared by {int(redundancy)} concepts total → does not count toward unique_rate"
                if redundancy is not None
                else "n/a"
            )
            + f" (method aggregate: {agg['unique_rate']:.2f}).",
            f"- **unk_rate** — this tree "
            + (f"has {n_uncovered} uncovered branch(es) → counts toward unk_rate" if n_uncovered else "has no uncovered branch → does not count toward unk_rate")
            + f" (method aggregate: {agg['unk_rate']:.2f}).",
            f"- **uncovered_rate** — this tree found "
            + (
                "zero tokens anywhere → not represented at all → counts toward uncovered_rate"
                if n_tokens == 0
                else f"{n_tokens} token(s) somewhere in the tree → does not count toward uncovered_rate"
                + (" (even though it still counts toward unk_rate above)" if n_uncovered else "")
            )
            + f" (method aggregate: {agg['uncovered_rate']:.2f}).",
            f"- **tree_complexity** — {n_contexts} context(s) (root + subcontexts) "
            f"(method aggregate: {agg['tree_complexity']:.2f}).",
            f"- **exact_rate** — this concept "
            + ("IS itself a selected token (0-hop)" if is_exact else "is NOT itself a selected token")
            + f" → {'counts' if is_exact else 'does not count'} toward exact_rate "
            f"(method aggregate: {agg['exact_rate']:.2f}).",
            f"- **semantic_coverage** — S_D(c,T) = "
            + (f"{frac_sem_cov:.2f}" if frac_sem_cov is not None else "n/a")
            + f" for this concept (method aggregate F_D(T): {agg['semantic_coverage']:.2f}).",
        ]
        display(Markdown("\n".join(lines)))
        display(Markdown("---"))


for tag, cid in example_concepts.items():
    explain_concept(cid, tag)

# check all results

In [ ]:
methods

## Takeaways

- `conciseness`/`tree_complexity`/`distance_score`/`unk_rate`/`uncovered_rate`/`exact_rate`/
  `unique_rate`/`semantic_coverage` are all just the mean (or fraction) of exactly the
  per-concept quantities printed above, taken over every concept in `M` -- nothing in
  `eval.py` computes anything these worked examples don't already show at the single-concept
  level.
- `uncovered_rate` is a *stricter* version of `unk_rate`: `unk_rate` counts a concept if *any*
  branch of its tree failed, `uncovered_rate` only counts it if *every* branch failed (zero
  tokens found anywhere) -- so `uncovered_rate <= unk_rate` always. A concept can have
  `unk_rate`'s uncovered=True on one branch while another branch still finds a token, which is
  exactly why the worked examples above sometimes show both bullets disagreeing.
- `uniqueness_entropy` is the one population-level metric: no single concept's tree "has" an
  entropy, but its `redundancy_group_size` (how many other concepts share its exact tree
  shape) is that concept's contribution to it -- a group size of 1 pushes entropy up, a large
  group pulls it down. `unique_rate` is the per-concept decomposition of exactly that signal:
  the fraction of concepts with `redundancy_group_size == 1`.
- Re-run the "Parameters" cell with a different `CONCEPT_IDS` / `K_EXAMPLE` /
  `CANDIDATE_LISTS` to build new worked examples -- e.g. compare `greedy_tree_margin` at a
  couple of different `lam` values, or pit `highest_degree` against `most_children`
  directly (`IMPLEMENTATION.md`'s tendency #5).